# fraud-v3-candidate — shadow run investigation

The model platform blocked the promotion (TESS-2310).

| | AUC | PR-AUC |
|---|---|---|
| candidate, offline (notebook 01) | **0.906** | 0.358 |
| candidate, shadow (2026-07-05 → 2026-08-27) | **0.699** | 0.071 |
| champion fraud-v2, same window | 0.778 | 0.164 |

Offline it beat the champion by a wide margin. In shadow it is *worse* than the champion. Why?

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
print(ROOT)

/Users/vishal/Documents/GitHub/fintech1/tessera-platform


In [2]:
report = json.loads((ROOT / "ml/registry/shadow/fraud-v3-candidate.json").read_text())
pd.DataFrame({
    "offline": report["offline"],
    "shadow": {k: v for k, v in report["shadow"].items() if k != "by_month"},
    "champion": report["champion_same_window"],
}).T

,auc,pr_auc,source
offline,0.9059,0.3578,"ml/notebooks/01_fraud_exploration.ipynb, cell 6"
shadow,0.6992,0.0706,NaN
champion,0.7782,0.164,NaN


In [3]:
df = pd.read_parquet(ROOT / "ml/data/transactions.parquet").sort_values("timestamp").reset_index(drop=True)
cut = df.timestamp.quantile(0.70)          # same temporal split as notebook 01
train, test = df[df.timestamp <= cut].copy(), df[df.timestamp > cut].copy()
print(f"train={len(train):,}  test={len(test):,}  test window {test.timestamp.min().date()} → {test.timestamp.max().date()}")
print("the shadow window is the test window")

train=42,000  test=18,000  test window 2026-07-05 → 2026-08-27
the shadow window is the test window


## Questions

1. Can the offline number be reproduced from notebook 01's feature set?
2. The report says `card_chargeback_rate` is non-zero for 27.4% of training rows but 14.2% of scored rows. Does the feature look the same at training time as at scoring time?
3. What does the candidate score without it?

## 1. Reproducing the offline number

Notebook 01 (cells 6 and 8) trains `HistGradientBoostingClassifier(max_iter=250, random_state=0)` on the earliest 70% of rows and scores the latest 30% — the same split as cell `load-data` above. Below is the same split, features, model and hyperparameters, checked against what the report says.

In [4]:
dev_counts = df.groupby("device_id").card_token.nunique()


def base_features(d: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "amount_log": np.log1p(d.amount_minor),
        "hour": d.timestamp.dt.hour,
        "is_night": ((d.timestamp.dt.hour >= 1) & (d.timestamp.dt.hour <= 5)).astype(int),
        "is_cross_border": d.is_cross_border,
        "mcc": d.mcc.astype("category").cat.codes,
        "device_card_count": d.device_id.map(dev_counts).fillna(1),
    }, index=d.index)


# Notebook 01 cell 8: one chargeback rate per card, taken over EVERY row in the table.
card_cb_rate = df.groupby("card_token").chargeback_filed_at.apply(lambda s: s.notna().mean())
offline_rate = df.card_token.map(card_cb_rate).fillna(0.0)


def with_rate(d: pd.DataFrame, rate: pd.Series) -> pd.DataFrame:
    """Base features plus one candidate version of card_chargeback_rate."""
    return base_features(d).assign(card_chargeback_rate=rate.loc[d.index])


def fit(X_train: pd.DataFrame) -> HistGradientBoostingClassifier:
    return HistGradientBoostingClassifier(max_iter=250, random_state=0).fit(X_train, train.is_fraud)


def score(model: HistGradientBoostingClassifier, X_test: pd.DataFrame) -> tuple[float, float]:
    p = model.predict_proba(X_test)[:, 1]
    return roc_auc_score(test.is_fraud, p), average_precision_score(test.is_fraud, p)


baseline_model = fit(base_features(train))
v3_model = fit(with_rate(train, offline_rate))
baseline_auc, baseline_ap = score(baseline_model, base_features(test))
v3_auc, v3_ap = score(v3_model, with_rate(test, offline_rate))

# If this ever stops matching, nothing below is grounded.
assert abs(v3_auc - report["offline"]["auc"]) < 5e-5 and abs(v3_ap - report["offline"]["pr_auc"]) < 5e-5
assert abs(baseline_auc - report["champion_same_window"]["auc"]) < 5e-5

print(f"lift of card_chargeback_rate over the v2 feature set: {v3_auc - baseline_auc:+.4f} AUC")
pd.DataFrame(
    {
        "AUC": [baseline_auc, v3_auc, report["champion_same_window"]["auc"], report["offline"]["auc"]],
        "PR-AUC": [baseline_ap, v3_ap, report["champion_same_window"]["pr_auc"], report["offline"]["pr_auc"]],
    },
    index=[
        "recomputed: v2 feature set (no card feature)",
        "recomputed: v3 candidate, notebook-01 method",
        "report: champion, same window",
        "report: v3 offline",
    ],
).round(4)

lift of card_chargeback_rate over the v2 feature set: +0.1277 AUC


,AUC,PR-AUC
recomputed: v2 feature set (no card feature),0.7782,0.1640
"recomputed: v3 candidate, notebook-01 method",0.9059,0.3578
"report: champion, same window",0.7782,0.1640
report: v3 offline,0.9059,0.3578


**Reproduced.** Notebook 01's method gives AUC 0.9059 / PR-AUC 0.3578 to four decimals. The no-feature baseline (0.7782 / 0.1640) is identical to the champion's same-window figures in the report, and the candidate differs from that baseline by exactly one feature — so the whole **+0.128 AUC** offline lift belongs to `card_chargeback_rate`.

The offline number is correct *as computed*. What has to be checked is whether the feature can be computed that way when a transaction is actually being scored.

## 2. Does `card_chargeback_rate` mean the same thing at training and scoring time?

Notebook 01 (cell 8) computes it with a `groupby` over the **whole table**: for a given row it counts every chargeback ever filed on that card — the row's own, those of the card's *later* transactions, and disputes filed weeks after the row. The report says serving cannot do that: the feature comes from the `card_history` nightly batch and counts *chargebacks filed as of scoring time*.

Those are different quantities, because disputes land 20–89 days after the transaction:

In [5]:
lag_days = (df.chargeback_filed_at - df.timestamp).dt.days
own_dispute_later = (test.chargeback_filed_at > test.timestamp).sum()
print(f"dispute lag: min {lag_days.min():.0f}, median {lag_days.median():.0f}, max {lag_days.max():.0f} days")
print(f"disputes on scored (test-window) rows filed after that row was scored: {own_dispute_later} of {test.chargeback_filed_at.notna().sum()}")
print(f"disputes in the table filed after its last transaction ({df.timestamp.max().date()}): "
      f"{(df.chargeback_filed_at > df.timestamp.max()).sum()} of {df.chargeback_filed_at.notna().sum()}")

dispute lag: min 20, median 54, max 89 days
disputes on scored (test-window) rows filed after that row was scored: 547 of 547
disputes in the table filed after its last transaction (2026-08-27): 561 of 1777


Now build the serving-time definition — for a row at time *t*: disputes on the card with `chargeback_filed_at <= t`, divided by the card's earlier transactions — and compare it with the two non-zero shares in the report.

In [6]:
def card_chargeback_rate_asof(d: pd.DataFrame) -> pd.Series:
    """Serving-time definition: at each row's timestamp, the card's disputes already *filed*
    divided by the card's *earlier* transactions. A row's own dispute can never count."""
    out = pd.Series(0.0, index=d.index)
    for _, g in d.groupby("card_token"):
        # timestamp and chargeback_filed_at are stored with different datetime units; force one
        t = g.timestamp.to_numpy("datetime64[ns]")
        filed = np.sort(g.chargeback_filed_at.dropna().to_numpy("datetime64[ns]"))
        n_filed = np.searchsorted(filed, t, side="right")       # disputes filed at or before t
        n_prior = np.searchsorted(np.sort(t), t, side="left")   # transactions strictly before t
        out.loc[g.index] = np.where(n_prior > 0, n_filed / np.maximum(n_prior, 1), 0.0)
    return out


asof_rate = card_chargeback_rate_asof(df)

stats = report["feature_stats"]["card_chargeback_rate"]
shares = pd.DataFrame(
    {
        "recomputed": [(offline_rate.loc[train.index] > 0).mean(), (asof_rate.loc[test.index] > 0).mean()],
        "shadow report": [stats["training_nonzero_share"], stats["shadow_nonzero_share"]],
    },
    index=["training rows, notebook-01 feature", "scored rows, as-of feature"],
)
assert (shares.recomputed - shares["shadow report"]).abs().max() < 5e-5
print(f"(for reference: notebook-01 feature on the test rows: {(offline_rate.loc[test.index] > 0).mean():.4f} non-zero — "
      "offline the feature looked the same on train and test, which is why nothing flagged it)")
shares.round(4)

(for reference: notebook-01 feature on the test rows: 0.2747 non-zero — offline the feature looked the same on train and test, which is why nothing flagged it)


,recomputed,shadow report
"training rows, notebook-01 feature",0.2743,0.2743
"scored rows, as-of feature",0.1419,0.1419


Both of the report's shares are reproduced exactly, so the as-of feature is what serving computes. Now: **which part of the notebook-01 feature carries the lift?** Peel the leak off one layer at a time. Each variant is used in *both* train and test and the model retrained, so the score is what a consistently-computed version of the feature is worth.

| | definition |
|---|---|
| **A** | notebook 01: every row of the table |
| **B** | A minus the row's own dispute |
| **C** | only the card's earlier transactions, but a dispute counts even if it is filed after *t* |
| **D** | only disputes already filed at *t* — what serving can know |

In [7]:
by_card = df.groupby("card_token")
disputed = df.chargeback_filed_at.notna().astype(int)
n_card = by_card.timestamp.transform("size")
n_disputed = disputed.groupby(df.card_token).transform("sum")
n_before = by_card.cumcount()                                            # df is sorted by timestamp
disputed_before = disputed.groupby(df.card_token).cumsum() - disputed

variants = {
    "A  notebook 01: every row of the table": offline_rate,
    "B  A minus the row's own dispute": ((n_disputed - disputed) / (n_card - 1).where(n_card > 1)).fillna(0.0),
    "C  earlier transactions only, any filing date": (disputed_before / n_before.where(n_before > 0)).fillna(0.0),
    "D  disputes filed by scoring time (serving)": asof_rate,
}

rows = {}
for name, rate in variants.items():
    auc, ap = score(fit(with_rate(train, rate)), with_rate(test, rate))
    rows[name] = {
        "non-zero, train": (rate.loc[train.index] > 0).mean(),
        "non-zero, test": (rate.loc[test.index] > 0).mean(),
        "feature alone, AUC": roc_auc_score(test.is_fraud, rate.loc[test.index]),
        "model AUC": auc,
        "model PR-AUC": ap,
    }
rows["(no card feature)"] = {"model AUC": baseline_auc, "model PR-AUC": baseline_ap}
anatomy = pd.DataFrame(rows).T
anatomy.round(4)

,"non-zero, train","non-zero, test","feature alone, AUC",model AUC,model PR-AUC
A notebook 01: every row of the table,0.2743,0.2747,0.8481,0.9059,0.3578
B A minus the row's own dispute,0.2526,0.2529,0.5116,0.7764,0.1611
"C earlier transactions only, any filing date",0.0942,0.2194,0.5061,0.7778,0.1599
D disputes filed by scoring time (serving),0.0344,0.1419,0.4976,0.7775,0.1562
(no card feature),NaN,NaN,NaN,0.7782,0.1640


**Reading the table**

- **Only A has any signal.** Remove *just the row's own dispute* (B) and the feature alone drops from AUC 0.848 to 0.512, and the model from 0.906 to 0.776 — slightly below the 0.778 no-feature baseline. The lift is each row's own future chargeback reading itself back to the model.
- **C and D do nothing** (feature alone ≈ 0.50, model ≈ baseline). D is the only one a scorer can build; C still counts disputes that had not been filed yet. In this data a card's *other* disputes carry no signal.
- **The own dispute is never knowable at scoring time**: every one of the disputed rows in the scored window was filed 20–89 days after the row (cell above).
- The as-of feature is also far sparser in training (3.4% non-zero vs 14.2% when scored): the table starts 2026-03-01, so early rows have no filed disputes behind them yet.

The plain correlation with the label, on the training rows, tells the same story:

In [8]:
observable = base_features(train)
best = observable.corrwith(train.is_fraud.astype(float)).abs().idxmax()
pd.Series({
    "notebook-01 feature (whole table)": np.corrcoef(offline_rate.loc[train.index], train.is_fraud)[0, 1],
    "as-of feature": np.corrcoef(asof_rate.loc[train.index], train.is_fraud)[0, 1],
    f"best observable feature ({best})": observable[best].corr(train.is_fraud.astype(float)),
}, name="corr with is_fraud, training rows").round(3)

notebook-01 feature (whole table)       0.285
as-of feature                           0.002
best observable feature (amount_log)    0.160
Name: corr with is_fraud, training rows, dtype: float64

## 3. Reproducing the shadow number

The candidate is *trained* on feature A — the only version notebook 01 can produce — but at scoring time it is *fed* feature D. If that is the whole story, scoring the notebook-01 model with the as-of feature on the shadow window should reproduce the shadow report.

In [9]:
X_serving = with_rate(test, asof_rate)
p_serving = v3_model.predict_proba(X_serving)[:, 1]
shadow_auc = roc_auc_score(test.is_fraud, p_serving)
shadow_ap = average_precision_score(test.is_fraud, p_serving)

month = test.timestamp.dt.strftime("%Y-%m")
months = sorted(month.unique())
recon_month = {m: roc_auc_score(test.is_fraud[month == m], p_serving[(month == m).to_numpy()]) for m in months}
rep_month = {r["month"]: r for r in report["shadow"]["by_month"]}
assert {m: int((month == m).sum()) for m in months} == {m: rep_month[m]["n"] for m in months}

recon = pd.DataFrame(
    {
        "reconstructed": [shadow_auc, shadow_ap] + [recon_month[m] for m in months],
        "shadow report": [report["shadow"]["auc"], report["shadow"]["pr_auc"]] + [rep_month[m]["auc"] for m in months],
    },
    index=["AUC", "PR-AUC"] + [f"AUC {m} (n={int((month == m).sum()):,})" for m in months],
)
assert (recon.reconstructed - recon["shadow report"]).abs().max() < 5e-4     # report gives months to 3 decimals
recon.round(4)

,reconstructed,shadow report
AUC,0.6992,0.6992
PR-AUC,0.0706,0.0706
"AUC 2026-07 (n=9,074)",0.7060,0.7060
"AUC 2026-08 (n=8,926)",0.6946,0.6950


In [10]:
gap = v3_auc - shadow_auc
lift_never_existed = v3_auc - baseline_auc
serving_damage = baseline_auc - shadow_auc
print(f"offline {v3_auc:.4f}  ->  shadow {shadow_auc:.4f}     gap {gap:.4f} AUC")
print(f"  lift that never existed (offline - no-feature baseline)      : {lift_never_existed:.4f}  ({lift_never_existed / gap:.0%})")
print(f"  damage from serving a model trained on the leak (baseline - shadow): {serving_damage:.4f}  ({serving_damage / gap:.0%})")

offline 0.9059  ->  shadow 0.6992     gap 0.2067 AUC
  lift that never existed (offline - no-feature baseline)      : 0.1277  (62%)
  damage from serving a model trained on the leak (baseline - shadow): 0.0790  (38%)


The reconstruction matches the report on every figure, so this is the mechanism. That still leaves a question: why is the candidate *worse* than the champion, rather than just level with it?

In [11]:
y = test.is_fraud.to_numpy()
offline_nz = (offline_rate.loc[test.index] > 0).to_numpy()
serving_nz = (asof_rate.loc[test.index] > 0).to_numpy()
p_offline = v3_model.predict_proba(with_rate(test, offline_rate))[:, 1]
p_baseline = baseline_model.predict_proba(base_features(test))[:, 1]

# What the feature says about fraud, offline vs when scored
display(pd.DataFrame(
    {
        "offline (A)": [y[offline_nz].mean(), y[~offline_nz].mean()],
        "serving (D)": [y[serving_nz].mean(), y[~serving_nz].mean()],
    },
    index=["fraud rate where feature > 0", "fraud rate where feature = 0"],
).round(4))

# What the candidate does with it once it is served
k = int(0.05 * len(test))                                  # alert budget: top 5% of transactions
top = np.argsort(-p_serving, kind="stable")[:k]


def caught(p: np.ndarray) -> int:
    return int(y[np.argsort(-p, kind="stable")[:k]].sum())


print(f"mean candidate score when served: feature > 0 -> {p_serving[serving_nz].mean():.4f}, "
      f"feature = 0 -> {p_serving[~serving_nz].mean():.4f}")
print(f"top-{k:,} alerts with a non-zero feature: {serving_nz[top].mean():.1%}   (base rate {serving_nz.mean():.1%})")
pd.Series(
    {
        "v3 offline (feature A)": caught(p_offline),
        "v2 feature set": caught(p_baseline),
        "v3 as shadowed (trained A, served D)": caught(p_serving),
    },
    name=f"frauds caught in the top {k:,} alerts (of {int(y.sum())})",
)

,offline (A),serving (D)
fraud rate where feature > 0,0.1147,0.0345
fraud rate where feature = 0,0.0052,0.0354


mean candidate score when served: feature > 0 -> 0.1112, feature = 0 -> 0.0069
top-900 alerts with a non-zero feature: 96.9%   (base rate 14.2%)


v3 offline (feature A)                  310
v2 feature set                          170
v3 as shadowed (trained A, served D)     69
Name: frauds caught in the top 900 alerts (of 635), dtype: int64

## 4. What does the candidate score without it?

In [12]:
d_auc, d_ap = anatomy.loc["D  disputes filed by scoring time (serving)", ["model AUC", "model PR-AUC"]]
# Difference of the unrounded AUCs. Subtracting the rounded table values below gives -0.0007.
print(f"honest lift of the as-of feature over the v2 feature set: {d_auc - baseline_auc:+.4f} AUC")
pd.DataFrame(
    [
        ("champion fraud-v2, same window (report)", report["champion_same_window"]["auc"], report["champion_same_window"]["pr_auc"]),
        ("candidate without card_chargeback_rate", baseline_auc, baseline_ap),
        ("candidate with the as-of feature, trained and served consistently", d_auc, d_ap),
        ("candidate as shadowed (trained on A, served D)", shadow_auc, shadow_ap),
        ("candidate offline (not achievable when scoring)", v3_auc, v3_ap),
    ],
    columns=["configuration", "AUC", "PR-AUC"],
).set_index("configuration").round(4)

honest lift of the as-of feature over the v2 feature set: -0.0008 AUC


,AUC,PR-AUC
configuration,,
"champion fraud-v2, same window (report)",0.7782,0.1640
candidate without card_chargeback_rate,0.7782,0.1640
"candidate with the as-of feature, trained and served consistently",0.7775,0.1562
"candidate as shadowed (trained on A, served D)",0.6992,0.0706
candidate offline (not achievable when scoring),0.9059,0.3578


Without the feature the candidate *is* the v2 feature set: 0.7782 / 0.1640, identical to the champion. With an honestly-computed version it scores 0.7775 / 0.1562 — an honest lift of **−0.0008 AUC**, within noise of zero. There is no evidence in this data that the feature adds anything a scorer can use.

## 5. Other explanations checked

**Label maturity.** The report says labels are "confirmed fraud as of 2026-09-15", and disputes take up to 89 days. If the shadow labels came from disputes filed by that date, the window would be badly under-labelled:

In [13]:
matured = (test.chargeback_filed_at <= pd.Timestamp("2026-09-15 23:59:59", tz="UTC")).astype(int).to_numpy()
print(f"fraud rows in the window: {int(y.sum())}; with a dispute filed by 2026-09-15: {int(matured.sum())} ({matured.sum() / y.sum():.0%})")
pd.DataFrame(
    {
        "AUC, true label": [roc_auc_score(y, p) for p in (p_baseline, p_serving)],
        "AUC, dispute-by-2026-09-15 label": [roc_auc_score(matured, p) for p in (p_baseline, p_serving)],
    },
    index=["v2 feature set (= champion figures)", "v3 as shadowed"],
).round(4)

fraud rows in the window: 635; with a dispute filed by 2026-09-15: 212 (33%)


,"AUC, true label","AUC, dispute-by-2026-09-15 label"
v2 feature set (= champion figures),0.7782,0.7500
v3 as shadowed,0.6992,0.6781


Both models would score lower under immature labels, and the candidate would still trail. More to the point, the report's champion figure (0.7782) matches the fully-labelled number, not the dispute-derived one (0.7500) — so the report was not using immature labels, and label maturity does not explain the gap.

**The other whole-table feature.** `device_card_count` in notebook 01 is also computed over all 180 days, future included. Making it as-of:

In [14]:
first_use = df.groupby(["device_id", "card_token"]).timestamp.transform("min")
cards_on_device_asof = (df.timestamp == first_use).astype(int).groupby(df.device_id).cumsum()   # distinct cards seen so far
X_tr = base_features(train).assign(device_card_count=cards_on_device_asof.loc[train.index])
X_te = base_features(test).assign(device_card_count=cards_on_device_asof.loc[test.index])
dev_auc, dev_ap = score(fit(X_tr), X_te)
print(f"v2 feature set with an as-of device_card_count: AUC {dev_auc:.4f} / PR-AUC {dev_ap:.4f}   "
      f"(whole-table version: {baseline_auc:.4f} / {baseline_ap:.4f}; change {dev_auc - baseline_auc:+.4f} AUC)")

v2 feature set with an as-of device_card_count: AUC 0.7739 / PR-AUC 0.1630   (whole-table version: 0.7782 / 0.1640; change -0.0043 AUC)


A real but small effect (about 2% of the gap): worth fixing in the same pass, not what broke v3.

## Conclusion

**The 0.906 was never a number that scoring time could achieve.** It reproduces exactly, but only because `card_chargeback_rate` in notebook 01 is computed over the whole table, so every row carries its own chargeback — the label — filed 20–89 days after the transaction it describes.

| | AUC |
|---|---|
| offline, notebook 01 | 0.9059 |
| − lift that never existed (leaked feature) | 0.7782 (= champion) |
| − damage from serving a model trained on the leak | **0.6992** (= shadow) |

- **Evidence.** Removing only the row's own dispute takes the model from 0.906 to 0.776. The serving-time definition reproduces the report's non-zero shares exactly (27.4% → 14.2%) and, fed to the notebook-01 model, its shadow scores exactly (AUC 0.6992, PR-AUC 0.0706; July 0.706, August 0.695).
- **Why worse than the champion, not just level.** The model learned "card has a dispute ⇒ fraud" (fraud rate 11.5% vs 0.5% offline). Served, the flag fires on 14% of rows that are no more fraudulent (3.45% vs 3.54%), yet the model scores them 16× higher: 97% of its top-5% alerts are rows with a prior card dispute, and it catches 69 of 635 frauds where the v2 feature set catches 170.
- **No lift once the leak is gone.** 0.7782 without the feature, 0.7775 with an as-of version, against the champion's 0.7782 — an honest lift of −0.0008 AUC. Nothing here supports promotion; TESS-2310 should stay blocked.

**Next steps**

1. Build the card history as `ml/features/aggregates.py` (the README lists it as missing) with an explicit `as_of` argument that only reads disputes with `chargeback_filed_at <= as_of`, and add a test that fails if a feature uses anything later.
2. Evaluate offline with the same as-of features the scorer will get, so offline and shadow cannot diverge this way. Fix `device_card_count` in the same pass (−0.004 AUC).
3. As-of history needs history. The as-of feature is non-zero for only 3.4% of training rows against 14.2% when scored, because the table starts on 2026-03-01 and disputes take 20–89 days to land. Backfill pre-window disputes before retraining.

*Caveat.* In this synthetic data a card carries no risk of its own (feature-alone AUC 0.51 / 0.51 / 0.50 for B–D), so a correct as-of feature is not expected to help *here*. On real data it might; the leak-free build above is how to find out.